# WorkflowContext from raw data

Ce notebook initialise un `WorkflowContext` a partir d'un dossier de fichiers bruts, puis fournit quelques cellules utiles pour inspecter `runs`, `configurations`, `issues` et generer les fichiers `refs_sub` / `refs_norm`.

In [37]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent

os.chdir(ROOT)
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"Repository root: {ROOT}")
print(f"Python path includes: {SRC}")

Repository root: /home/achennev/python/scarlet
Python path includes: /home/achennev/python/scarlet/src


In [38]:
from scarlet.workflow.context import initialize_workflow_context_from_raw_directory

RAW_DIR = ROOT / "data" / "SANSLLB" / "raw"
OUTPUT_DIR = ROOT / "data" / "SANSLLB" / "out"
INSTRUMENT_NAME = "sansllb"

RAW_DIR, OUTPUT_DIR

(PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw'),
 PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out'))

In [39]:
w = initialize_workflow_context_from_raw_directory(
    RAW_DIR,
    instrument_name=INSTRUMENT_NAME,
    output_dir=OUTPUT_DIR,
    overwrite=True
)

print(f"runs: {len(w.runs)}")
print(f"configurations: {len(w.configurations)}")
print(f"issues: {len(w.issues)}")
print(f"artifacts: {len(w.artifacts)}")

runs: 62
configurations: 2
issues: 4
artifacts: 64


In [40]:
w.runs_table()

sample_name,config_id,mode,entity,file_path
empty_beam_att5_ws_beamstop,config_1,transmission,empty_beam,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs
empty_cell,config_1,transmission,empty_cell,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs
H2O,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002342.nxs
S1_P_PB_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002343.nxs
S2_P_PB_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002344.nxs
S3_P_PBK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002345.nxs
S4_P_BC_25_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002346.nxs
S5_P_BC_60_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002347.nxs
S6_P_BCK_40_2_2mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002348.nxs
S7_P_MI_25_1mm,config_1,transmission,sample,/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002349.nxs


In [41]:
w.configurations_table()

config_id,wavelength,sample_detector_distance,notes,has_collimation,collimation_distance,last_aperture_to_sample_distance,aperture1_type,aperture1_x_gap,aperture1_y_gap,aperture1_diameter,aperture2_type,aperture2_x_gap,aperture2_y_gap,aperture2_diameter
config_1,6.00046,2.5,,True,0.5,1.2,slit,0.0419996,0.0419996,,slit,0.0319997,0.0320003,
config_2,5.9989,9,,True,2.58333,1.2,slit,0.0269997,0.0269999,,slit,0.0169996,0.0169998,


In [42]:
from scarlet.workflow.context import generate_reference_files_from_workflow_context

generate_reference_files_from_workflow_context(w)
print("refs_sub:", {k: str(v) for k, v in sorted(w.refs_sub_files.items())})
print("refs_norm:", {k: str(v) for k, v in sorted(w.refs_norm_files.items())})

refs_sub: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_sub_config_2.nxs'}
refs_norm: {'config_1': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs', 'config_2': '/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs'}


In [43]:
from scarlet.workflow import update_detector0_beam_center_from_empty_beam_transmission
for key in w.refs_sub_files:
    update_detector0_beam_center_from_empty_beam_transmission(w.refs_sub_files[key])

In [44]:
from scarlet.gui import run_mask_editor

# run_mask_editor()

In [45]:
# from scarlet.workflow.reference import insert_masks_in_refs_file
# for key in w.refs_norm_files:
#     insert_masks_in_refs_file(w.refs_norm_files[key], w.masks)
# for key in w.refs_sub_files:
#     insert_masks_in_refs_file(w.refs_sub_files[key], w.masks)   

In [46]:
from scarlet.workflow.context import write_runs_report_csv

In [47]:
file = write_runs_report_csv(w, overwrite=True)
print(f"Runs report written to: {file}")

Runs report written to: /home/achennev/python/scarlet/data/SANSLLB/out/runs_report.csv


In [48]:
# test change with respect to modified run csv file
# from scarlet.workflow.context import update_workflow_context_from_runs_report_csv
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report_modified.csv")
# update_workflow_context_from_runs_report_csv(w,"data/SANSLLB/out/runs_report.csv")
# file = write_runs_report_csv(w, csv_path="data/SANSLLB/out/runs_report_updated.csv", overwrite=True)

In [49]:
from scarlet.reduction.transmission import compute_reference_transmissions
for key in w.refs_norm_files:
    compute_reference_transmissions(w.refs_norm_files[key])
for key in w.refs_sub_files:
    compute_reference_transmissions(w.refs_sub_files[key])


In [50]:
w.refs_norm_files

{'config_1': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_1.nxs'),
 'config_2': PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/refs_norm_config_2.nxs')}

In [ ]:
from scarlet.workflow import update_reference_masks_from_workflow_context

w = update_reference_masks_from_workflow_context(w)

{('config_1',
  0): array([[1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], shape=(128, 128), dtype=uint8),
 ('config_1',
  1): array([[1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        ...,
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1]], shape=(136, 16), dtype=uint8),
 ('config_1',
  2): array([[1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        ...,
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1],
        [1, 1, 1, ..., 1, 1, 1]], shape=(16, 136), dtype=uint8),
 ('config_2',
  0): array([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0

In [52]:
# from scarlet.workflow.reference import write_corrected_water_scattering
# write_corrected_water_scattering(w.refs_norm_files['config_1'])

In [53]:
# Save workflow context to file
from scarlet.workflow import save_workflow_context, load_workflow_context

save_workflow_context(w, "workflow_context.nxs")
w2 = load_workflow_context("workflow_context.nxs")

In [54]:
from scarlet.workflow.context import update_transmissions_from_workflow_context
update_transmissions_from_workflow_context(w)
w.transmissions_table()

sample_name,config_id,transmission
H2O,config_1,0.545957
H2O,config_2,0.592539
S10_P_BB_60_1mm,config_1,0.874308
S10_P_BB_60_1mm,config_2,0.986359
S11_P_EB_25_1mm,config_1,0.885527
S11_P_EB_25_1mm,config_2,0.998175
S12_P_EB_60_1mm,config_1,0.878189
S12_P_EB_60_1mm,config_2,0.983866
S1_P_PB_25_2mm,config_1,0.826591
S1_P_PB_25_2mm,config_2,0.925727


In [55]:
from scarlet.workflow import integrate_scattering_from_workflow_context

integrate_scattering_from_workflow_context(w, n_bins=200)

WorkflowContext(experiment_id='experiment', instrument_name='sansllb', root_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/raw'), output_dir=PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out'), schema_raw='scarlet_nxsas_raw_v1.3_mono.yaml', schema_refs_sub='scarlet_refs_sub_v1.0.yaml', schema_refs_norm='scarlet_refs_norm_v1.0.yaml', schema_masks='scarlet_masks_v1.0.yaml', runs={RunKey(config_id='config_1', entity='empty_beam', mode='transmission', sample_name='empty_beam_att5_ws_beamstop'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002339.nxs'), RunKey(config_id='config_1', entity='dark', mode='scattering', sample_name='Cd'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002356.nxs'), RunKey(config_id='config_1', entity='empty_cell', mode='transmission', sample_name='empty_cell'): PosixPath('/home/achennev/python/scarlet/data/SANSLLB/out/sans-llb2025n002341.nxs'), RunKey(config_id='config_1', entity='sample', mode